# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [23]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

!pip install numpy pandas seaborn matplotlib scikit-learn tqdm

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))




[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [24]:
# Add as many cells as you need

df = pd.read_csv("zillow_cleaned.csv")

In [25]:
df

,airconditioningtypeid,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedbathnbr,calculatedfinishedsquarefeet,finishedsquarefeet12,fips,fullbathcnt,garagecarcnt,...,propertyzoningdesc_WVRPD112U*,propertyzoningdesc_WVRPD12000,propertyzoningdesc_WVRPD12U*,propertyzoningdesc_WVRPD12U-R,propertyzoningdesc_WVRPD17U*,propertyzoningdesc_WVRPD18U*,propertyzoningdesc_WVRPD40000,propertyzoningdesc_WVRPD4OOOO,propertyzoningdesc_WVRR,propertyzoningdesc_WVRR1-RPD1
0,1.0,3.5,4.0,6.0,3.5,3100.0,3100.0,6059.0,3.0,2.0,...,False,False,False,False,False,False,False,False,False,False
1,1.0,1.0,2.0,6.0,1.0,1465.0,1465.0,6111.0,1.0,1.0,...,False,False,False,False,False,False,False,False,False,False
2,1.0,2.0,3.0,6.0,2.0,1243.0,1243.0,6059.0,2.0,2.0,...,False,False,False,False,False,False,False,False,False,False
3,1.0,3.0,4.0,8.0,3.0,2376.0,2376.0,6037.0,3.0,2.0,...,False,False,False,False,False,False,False,False,False,False
4,1.0,3.0,3.0,8.0,3.0,1312.0,1312.0,6037.0,3.0,2.0,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72181,1.0,3.0,3.0,8.0,3.0,1741.0,1741.0,6037.0,3.0,2.0,...,False,False,False,False,False,False,False,False,False,False
72182,1.0,2.0,2.0,6.0,2.0,1286.0,1286.0,6037.0,2.0,2.0,...,False,False,False,False,False,False,False,False,False,False
72183,1.0,2.0,4.0,6.0,2.0,1612.0,1612.0,6111.0,2.0,2.0,...,False,False,False,False,False,False,False,False,False,False
72184,1.0,1.0,3.0,4.0,1.0,1032.0,1032.0,6037.0,1.0,2.0,...,False,False,False,False,False,False,False,False,False,False


In [26]:
df = df.select_dtypes(include=[np.number])
df = df.dropna()

In [27]:
X = df.drop(columns=["taxvaluedollarcnt"])
y = df["taxvaluedollarcnt"]

In [28]:
X

,airconditioningtypeid,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedbathnbr,calculatedfinishedsquarefeet,finishedsquarefeet12,fips,fullbathcnt,garagecarcnt,...,lotsizesquarefeet,propertylandusetypeid,regionidcity,regionidcounty,regionidneighborhood,regionidzip,roomcnt,unitcnt,yearbuilt,numberofstories
0,1.0,3.5,4.0,6.0,3.5,3100.0,3100.0,6059.0,3.0,2.0,...,4506.0,261.0,53571.0,1286.0,118849.0,96978.0,0.0,1.0,1998.0,1.0
1,1.0,1.0,2.0,6.0,1.0,1465.0,1465.0,6111.0,1.0,1.0,...,12647.0,261.0,13091.0,2061.0,118849.0,97099.0,5.0,1.0,1967.0,1.0
2,1.0,2.0,3.0,6.0,2.0,1243.0,1243.0,6059.0,2.0,2.0,...,8432.0,261.0,21412.0,1286.0,118849.0,97078.0,6.0,1.0,1962.0,1.0
3,1.0,3.0,4.0,8.0,3.0,2376.0,2376.0,6037.0,3.0,2.0,...,13038.0,261.0,396551.0,3101.0,118849.0,96330.0,0.0,1.0,1970.0,1.0
4,1.0,3.0,3.0,8.0,3.0,1312.0,1312.0,6037.0,3.0,2.0,...,278581.0,266.0,12447.0,3101.0,268548.0,96451.0,0.0,1.0,1964.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72181,1.0,3.0,3.0,8.0,3.0,1741.0,1741.0,6037.0,3.0,2.0,...,59487.0,266.0,12447.0,3101.0,32368.0,96415.0,0.0,1.0,1980.0,1.0
72182,1.0,2.0,2.0,6.0,2.0,1286.0,1286.0,6037.0,2.0,2.0,...,47405.0,261.0,12447.0,3101.0,27328.0,96284.0,0.0,1.0,1940.0,1.0
72183,1.0,2.0,4.0,6.0,2.0,1612.0,1612.0,6111.0,2.0,2.0,...,12105.0,261.0,27110.0,2061.0,118849.0,97116.0,7.0,1.0,1964.0,1.0
72184,1.0,1.0,3.0,4.0,1.0,1032.0,1032.0,6037.0,1.0,2.0,...,5074.0,261.0,36502.0,3101.0,118849.0,96480.0,0.0,1.0,1954.0,1.0


In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [30]:
# Add as many cells as you need

X_train_scaled_pre = np.asarray(X_train_scaled, dtype=np.float32)
X_test_scaled_pre = np.asarray(X_test_scaled, dtype=np.float32)
y_train_pre = y_train.to_numpy(dtype=np.float32)
y_test_pre = y_test.to_numpy(dtype=np.float32)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

In [31]:
# Add as many cells as you need

X_train_scaled = np.asarray(X_train_scaled, dtype=np.float32)
X_test_scaled = np.asarray(X_test_scaled, dtype=np.float32)
y_train = y_train.to_numpy(dtype=np.float32)
y_test = y_test.to_numpy(dtype=np.float32)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

In [32]:
def evaluate_model(model, X, y, cv):
    mae_scores = []

    for train_idx, val_idx in cv.split(X):
        X_fold_train = X[train_idx]
        X_fold_val = X[val_idx]
        y_fold_train = y[train_idx]
        y_fold_val = y[val_idx]

        model.fit(X_fold_train, y_fold_train)
        preds = model.predict(X_fold_val)
        mae = mean_absolute_error(y_fold_val, preds)
        mae_scores.append(mae)

    return np.mean(mae_scores), np.std(mae_scores)

In [33]:
## run the cross validation mae

from sklearn.metrics import mean_absolute_error

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(max_iter=5000)
}

results = {}

for name, model in models.items():
    mean_mae, std_mae = evaluate_model(model, X_train_scaled_pre, y_train_pre, cv)
    results[name] = (mean_mae, std_mae)

for name, (mean_mae, std_mae) in results.items():
    print(f"{name}:")
    print(f"Mean CV MAE: {mean_mae:.2f}")
    print(f"Std CV MAE: {std_mae:.2f}")
    print()

Linear Regression:
Mean CV MAE: 151662.60
Std CV MAE: 1128.86

Ridge Regression:
Mean CV MAE: 151662.72
Std CV MAE: 1128.85

Lasso Regression:
Mean CV MAE: 151662.75
Std CV MAE: 1128.85



### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

### Part 1: Conclusion

All three models, ridge, lasso, and standard linear regression, performed nearly identically in terms of both mean MAE and standard deviation. This suggests that their predictive strength and stability are very similar. Notably, ridge and lasso regression are typically beneficial when there is strong multicollinearity among features, as they apply regularization to stabilize coefficients and improve feature selection. The fact that their performance closely mirrors that of ordinary linear regression indicates that multicollinearity is likely not a significant issue in this dataset, or that the existing feature set is already well-conditioned.

Additionally, the lack of performance improvement from regularization suggests that coefficient magnitudes are already reasonable and are not being meaningfully shrunk by ridge or lasso penalties. Overall, the comparable performance across all three models implies that the feature set is already informative, the model is not overfitting, and additional regularization or feature selection may not provide substantial benefit at this stage.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [34]:
# Add as many cells as you need

## add engineered features (3)


# 1. Home age
X_train["home_age"] = 2017 - X_train["yearbuilt"]
X_test["home_age"] = 2017 - X_test["yearbuilt"]

# 2. Bathroom-to-bedroom ratio
X_train["bath_bed_ratio"] = X_train["bathroomcnt"] / (X_train["bedroomcnt"] + 1)
X_test["bath_bed_ratio"] = X_test["bathroomcnt"] / (X_test["bedroomcnt"] + 1)

# 3. Square feet per room
X_train["sqft_per_room"] = X_train["calculatedfinishedsquarefeet"] / (X_train["roomcnt"] + 1)
X_test["sqft_per_room"] = X_test["calculatedfinishedsquarefeet"] / (X_test["roomcnt"] + 1)

In [35]:
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

In [36]:
## scale features again

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = np.asarray(X_train_scaled, dtype=np.float32)
X_test_scaled = np.asarray(X_test_scaled, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.float32)

In [37]:
## rerun the 3 models with repeated cv

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

def evaluate_model(model, X, y, cv):
    mae_scores = []

    for train_idx, val_idx in cv.split(X):
        X_fold_train = X[train_idx]
        X_fold_val = X[val_idx]
        y_fold_train = y[train_idx]
        y_fold_val = y[val_idx]

        model.fit(X_fold_train, y_fold_train)
        preds = model.predict(X_fold_val)
        mae_scores.append(mean_absolute_error(y_fold_val, preds))

    return np.mean(mae_scores), np.std(mae_scores)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(max_iter=5000)
}

for name, model in models.items():
    mean_mae, std_mae = evaluate_model(model, X_train_scaled, y_train, cv)
    print(f"{name}:")
    print(f"Mean CV MAE: {mean_mae:.2f}")
    print(f"Std CV MAE: {std_mae:.2f}")
    print()

Linear Regression:
Mean CV MAE: 151680.82
Std CV MAE: 1130.44

Ridge Regression:
Mean CV MAE: 151680.81
Std CV MAE: 1130.43

Lasso Regression:
Mean CV MAE: 151680.83
Std CV MAE: 1130.43



### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?



### Part 2: Conclusion

After introducing the new engineered features, none of the models, linear regression, ridge, or lasso, showed any meaningful improvement in performance. In fact, the mean cross,validated MAE slightly increased to approximately 151,680 for all three models, with nearly identical standard deviations (~1130). This indicates not only a lack of improvement, but also that the added features may have very slightly degraded model performance.

Because all three models continued to perform almost identically, there is no evidence that any specific engineered feature provided a meaningful benefit. If certain features had been useful, we would expect at least the regularized models (ridge or lasso) to respond differently by either improving performance or shrinking less relevant coefficients. However, the unchanged results suggest that the new features either did not add new predictive information or introduced redundancy.

One possible explanation is that the original feature set already captured most of the relevant signal in the data, making additional engineered features unnecessary. Another hypothesis is that the new features may be highly correlated with existing ones, providing little new information while potentially adding noise. This could explain the very slight increase in MAE. Additionally, since regularization did not improve performance, it reinforces the idea that the feature space was already well, behaved, with no severe multicollinearity or overfitting issues for the models to correct.

Overall, these results suggest that the feature engineering step did not enhance model performance.


### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [38]:
# Add as many cells as you need

## rank features

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

print(feature_importance)

                         feature  importance
5   calculatedfinishedsquarefeet    0.208198
12                      latitude    0.142744
13                     longitude    0.099039
6           finishedsquarefeet12    0.092971
14             lotsizesquarefeet    0.079474
19                   regionidzip    0.067377
22                     yearbuilt    0.055861
24                      home_age    0.055244
26                 sqft_per_room    0.043933
3          buildingqualitytypeid    0.027829
16                  regionidcity    0.024514
18          regionidneighborhood    0.024197
25                bath_bed_ratio    0.020431
10               garagetotalsqft    0.016845
2                     bedroomcnt    0.010931
11         heatingorsystemtypeid    0.005179
1                    bathroomcnt    0.004786
4              calculatedbathnbr    0.004677
15         propertylandusetypeid    0.003964
8                    fullbathcnt    0.002900
20                       roomcnt    0.002527
23        

In [39]:
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

def evaluate_model(model, X, y, cv):
    mae_scores = []

    for train_idx, val_idx in cv.split(X):
        X_fold_train = X[train_idx]
        X_fold_val = X[val_idx]
        y_fold_train = y[train_idx]
        y_fold_val = y[val_idx]

        model.fit(X_fold_train, y_fold_train)
        preds = model.predict(X_fold_val)
        mae_scores.append(mean_absolute_error(y_fold_val, preds))

    return np.mean(mae_scores), np.std(mae_scores)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(max_iter=5000)
}

# Try top 5, 10, 15, 20, and all features
subset_sizes = [5, 10, 15, 20]
subset_sizes = [k for k in subset_sizes if k <= len(feature_importance)]

best_results = {}

for model_name, model in models.items():
    best_mae = float("inf")
    best_std = None
    best_features = None

    for k in subset_sizes:
        selected_features = feature_importance["feature"].head(k).tolist()

        X_train_subset = X_train[selected_features]
        X_test_subset = X_test[selected_features]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_subset)
        X_test_scaled = scaler.transform(X_test_subset)

        X_train_scaled = np.asarray(X_train_scaled, dtype=np.float32)
        y_train_array = np.asarray(y_train, dtype=np.float32)

        mean_mae, std_mae = evaluate_model(model, X_train_scaled, y_train_array, cv)

        print(f"{model_name} | Top {k} features | Mean MAE: {mean_mae:.2f} | Std: {std_mae:.2f}")

        if mean_mae < best_mae:
            best_mae = mean_mae
            best_std = std_mae
            best_features = selected_features

    best_results[model_name] = {
        "features": best_features,
        "mean_mae": best_mae,
        "std_mae": best_std
    }

Linear Regression | Top 5 features | Mean MAE: 155624.26 | Std: 1112.58
Linear Regression | Top 10 features | Mean MAE: 152911.38 | Std: 1117.80
Linear Regression | Top 15 features | Mean MAE: 152111.03 | Std: 1096.26
Linear Regression | Top 20 features | Mean MAE: 151881.29 | Std: 1099.34
Ridge Regression | Top 5 features | Mean MAE: 155624.44 | Std: 1112.58
Ridge Regression | Top 10 features | Mean MAE: 152911.56 | Std: 1117.80
Ridge Regression | Top 15 features | Mean MAE: 152111.18 | Std: 1096.26
Ridge Regression | Top 20 features | Mean MAE: 151881.45 | Std: 1099.33
Lasso Regression | Top 5 features | Mean MAE: 155624.38 | Std: 1112.58
Lasso Regression | Top 10 features | Mean MAE: 152911.49 | Std: 1117.81
Lasso Regression | Top 15 features | Mean MAE: 152111.12 | Std: 1096.25
Lasso Regression | Top 20 features | Mean MAE: 151881.47 | Std: 1099.33


In [40]:
## print the best-performing subset for each model

for model_name, result in best_results.items():
    print(f"{model_name}:")
    print("Best features:")
    print(result["features"])
    print(f"Mean CV MAE: {result['mean_mae']:.2f}")
    print(f"Std CV MAE: {result['std_mae']:.2f}")
    print()

Linear Regression:
Best features:
['calculatedfinishedsquarefeet', 'latitude', 'longitude', 'finishedsquarefeet12', 'lotsizesquarefeet', 'regionidzip', 'yearbuilt', 'home_age', 'sqft_per_room', 'buildingqualitytypeid', 'regionidcity', 'regionidneighborhood', 'bath_bed_ratio', 'garagetotalsqft', 'bedroomcnt', 'heatingorsystemtypeid', 'bathroomcnt', 'calculatedbathnbr', 'propertylandusetypeid', 'fullbathcnt']
Mean CV MAE: 151881.29
Std CV MAE: 1099.34

Ridge Regression:
Best features:
['calculatedfinishedsquarefeet', 'latitude', 'longitude', 'finishedsquarefeet12', 'lotsizesquarefeet', 'regionidzip', 'yearbuilt', 'home_age', 'sqft_per_room', 'buildingqualitytypeid', 'regionidcity', 'regionidneighborhood', 'bath_bed_ratio', 'garagetotalsqft', 'bedroomcnt', 'heatingorsystemtypeid', 'bathroomcnt', 'calculatedbathnbr', 'propertylandusetypeid', 'fullbathcnt']
Mean CV MAE: 151881.45
Std CV MAE: 1099.33

Lasso Regression:
Best features:
['calculatedfinishedsquarefeet', 'latitude', 'longitude', 

### Part 3: Discussion [3 pts]


Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


### Part 3: Conclusion

Feature selection had little effect on the performance of any of the models. After reducing the original feature set, all three models, linear regression, ridge, and lasso, continued to perform almost identically in terms of mean cross-validated MAE and standard deviation. There was no clear performance improvement from eliminating features, and all models achieved nearly the same MAE of approximately 151,880 with very similar variability across folds. This suggests that reducing the number of features did not meaningfully improve or degrade model performance.

Interestingly, all three models selected the same subset of 20 features as their optimal configuration. This indicates that these features consistently captured the most relevant information in the data, regardless of whether regularization was applied. If feature selection or regularization had a stronger effect, we would expect the lasso model in particular to retain fewer features or produce noticeably different results, but this was not observed.

All three newly engineered features: bath_bed_ratio, sqft_per_room, and home_age, were selected by every model. This suggests that while feature selection did not lead to overall performance improvements, the engineered features were still considered informative enough to be retained alongside the original predictors.

Overall, these results suggest that the selected features already formed a stable and well-behaved feature set, leaving little room for feature selection or regularization to provide additional gains.

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [41]:
# Add as many cells as you need

tuning_cv = 3
final_cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

tuned_results = {}

selected_feature_sets = {
    "Linear Regression": best_results["Linear Regression"]["features"],
    "Ridge Regression": best_results["Ridge Regression"]["features"],
    "Lasso Regression": best_results["Lidge Regression"]["features"] if "Lidge Regression" in best_results else best_results["Lasso Regression"]["features"]
}

print("=" * 70)
print("Part 4 - Fine-Tuned Results")
print("=" * 70)

# loop through each model, and return the best hyperparameters, features, mean MAE, and std MAE for each
for model_name, features in selected_feature_sets.items():
    X_train_subset = X_train[features]

    scaler = StandardScaler()
    X_train_subset_scaled = scaler.fit_transform(X_train_subset)

    if model_name == "Linear Regression":
        model = LinearRegression()
        param_grid = {
            "fit_intercept": [True, False],
            "positive": [False]
        }

    elif model_name == "Ridge Regression":
        model = Ridge(random_state=42)
        param_grid = {
            "alpha": [0.01, 0.1, 1.0, 10.0, 100.0],
            "fit_intercept": [True, False]
        }

    elif model_name == "Lasso Regression":
        model = Lasso(random_state=42, max_iter=5000)
        param_grid = {
            "alpha": [0.0005, 0.001, 0.01, 0.1, 1.0],
            "fit_intercept": [True, False]
        }

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_absolute_error",
        cv=tuning_cv,
        n_jobs=-1
    )

    grid.fit(X_train_subset_scaled, y_train)

    best_model = grid.best_estimator_
    mean_mae, std_mae = evaluate_model(best_model, X_train_subset_scaled, y_train, final_cv)

    tuned_results[model_name] = {
        "features": features,
        "best_params": grid.best_params_,
        "mean_mae": mean_mae,
        "std_mae": std_mae
    }

    # print results for each model
    print(model_name)
    print("Best params:", grid.best_params_)
    print("Best features:", features)
    print(f"Mean CV MAE: {mean_mae:.2f}")
    print(f"Std CV MAE: {std_mae:.2f}")
    print("-" * 70)

Part 4 - Fine-Tuned Results


Linear Regression
Best params: {'fit_intercept': True, 'positive': False}
Best features: ['calculatedfinishedsquarefeet', 'latitude', 'longitude', 'finishedsquarefeet12', 'lotsizesquarefeet', 'regionidzip', 'yearbuilt', 'home_age', 'sqft_per_room', 'buildingqualitytypeid', 'regionidcity', 'regionidneighborhood', 'bath_bed_ratio', 'garagetotalsqft', 'bedroomcnt', 'heatingorsystemtypeid', 'bathroomcnt', 'calculatedbathnbr', 'propertylandusetypeid', 'fullbathcnt']
Mean CV MAE: 151880.37
Std CV MAE: 1257.53
----------------------------------------------------------------------
Ridge Regression
Best params: {'alpha': 0.01, 'fit_intercept': True}
Best features: ['calculatedfinishedsquarefeet', 'latitude', 'longitude', 'finishedsquarefeet12', 'lotsizesquarefeet', 'regionidzip', 'yearbuilt', 'home_age', 'sqft_per_room', 'buildingqualitytypeid', 'regionidcity', 'regionidneighborhood', 'bath_bed_ratio', 'garagetotalsqft', 'bedroomcnt', 'heatingorsystemtypeid', 'bathroomcnt', 'calculatedbathnbr',

In [43]:
## run the cross validation mae

from sklearn.metrics import mean_absolute_error

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(max_iter=5000)
}

results = {}

for name, model in models.items():
    mean_mae, std_mae = evaluate_model(model, X_train_scaled_pre, y_train_pre, cv)
    results[name] = (mean_mae, std_mae)

for name, (mean_mae, std_mae) in results.items():
    print(f"{name}:")
    print(f"Mean CV MAE: {mean_mae:.2f}")
    print(f"Std CV MAE: {std_mae:.2f}")
    print()

Linear Regression:
Mean CV MAE: 151662.60
Std CV MAE: 1128.86

Ridge Regression:
Mean CV MAE: 151662.72
Std CV MAE: 1128.85

Lasso Regression:
Mean CV MAE: 151662.75
Std CV MAE: 1128.85



### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


### Part 4: Conclusion

After evaluating all three models, we found that the original Linear, Ridge, and Lasso regressions performed best prior to applying feature engineering, feature selection, and hyperparameter tuning. For each model, we used cross‑validated grid search to tune model‑specific hyperparameters, focusing on regularization strength for Ridge and Lasso and intercept settings for Linear Regression, as these parameters directly control model complexity. We explored multiple preprocessing approaches, including adjusting the feature set through selection and engineered transformations, as well as tuning regularization hyperparameters for Ridge and Lasso. However, these modifications did not improve performance and in some cases slightly increased the cross‑validated MAE. This suggests that the original feature set already represented the underlying relationships in the data effectively, and that additional feature selection or regularization may have removed informative features or introduced unnecessary bias. Since all three models exhibited similar performance and low variance across folds, further tuning provided limited benefit. As a result, the baseline versions of the models were retained, as they produced the most stable and accurate results.

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [42]:
# Add as many cells as you need

# find the best tuned model from Part 4 (lowest mean CV MAE)
best_model_name = min(tuned_results, key=lambda name: tuned_results[name]["mean_mae"])
best_model_info = tuned_results[best_model_name]

best_features = best_model_info["features"]
best_params = best_model_info["best_params"]

print("=" * 70)
print("Final Model Selection")
print("=" * 70)
print("Best model from Part 4:", best_model_name)
print("Best parameters:", best_params)
print("Best features:", best_features)
print(f"Best CV MAE: {best_model_info['mean_mae']:.2f}")
print()

# subset training and test data to the winning feature set
X_train_best = X_train[best_features]
X_test_best = X_test[best_features]

# scale using training data only
scaler = StandardScaler()
X_train_best_scaled = scaler.fit_transform(X_train_best)
X_test_best_scaled = scaler.transform(X_test_best)

# rebuild the winning model using its best parameters
if best_model_name == "Linear Regression":
    final_model = LinearRegression(**best_params)

elif best_model_name == "Ridge Regression":
    final_model = Ridge(random_state=42, **best_params)

elif best_model_name == "Lasso Regression":
    final_model = Lasso(random_state=42, max_iter=5000, **best_params)

# fit on training data
final_model.fit(X_train_best_scaled, y_train)

# predict on test data
y_test_pred = final_model.predict(X_test_best_scaled)

# compute test MAE
test_mae = mean_absolute_error(y_test, y_test_pred)

print("=" * 70)
print("Test Set Results")
print("=" * 70)
print("Final selected model:", best_model_name)
print("Parameters used:", best_params)
print(f"Test MAE: {test_mae:.2f}")

Final Model Selection
Best model from Part 4: Linear Regression
Best parameters: {'fit_intercept': True, 'positive': False}
Best features: ['calculatedfinishedsquarefeet', 'latitude', 'longitude', 'finishedsquarefeet12', 'lotsizesquarefeet', 'regionidzip', 'yearbuilt', 'home_age', 'sqft_per_room', 'buildingqualitytypeid', 'regionidcity', 'regionidneighborhood', 'bath_bed_ratio', 'garagetotalsqft', 'bedroomcnt', 'heatingorsystemtypeid', 'bathroomcnt', 'calculatedbathnbr', 'propertylandusetypeid', 'fullbathcnt']
Best CV MAE: 151880.37

Test Set Results
Final selected model: Linear Regression
Parameters used: {'fit_intercept': True, 'positive': False}
Test MAE: 152258.17


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here